<a href="https://colab.research.google.com/github/jaywong8/Detection-of-Atrial-Fibrillation-Termination/blob/main/DatasetAuditingfor%20AFib%20Positive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install wfdb pandas numpy scipy scikit-learn matplotlib

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import time
import requests
import wfdb
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_URL = "https://physionet.org/files/icentia11k-continuous-ecg/1.0"
OUT_DIR = "/content/drive/MyDrive/icentia11k_afib_audit"
os.makedirs(OUT_DIR, exist_ok=True)

SEGMENT_CSV = os.path.join(OUT_DIR, "afib_positive_segments.csv")
ERROR_CSV = os.path.join(OUT_DIR, "audit_errors.csv")
CHECKPOINT_CSV = os.path.join(OUT_DIR, "audited_segments.csv")
PATIENT_CSV = os.path.join(OUT_DIR, "audited_patients.csv")

import threading
import requests
import time

_thread_local = threading.local()

def get_session():
    if not hasattr(_thread_local, "session"):
        s = requests.Session()
        adapter = requests.adapters.HTTPAdapter(
            pool_connections=2,
            pool_maxsize=2,
            max_retries=0
        )
        s.mount("https://", adapter)
        _thread_local.session = s
    return _thread_local.session

def fetch(url, retries=4):
    for attempt in range(retries):
        try:
            r = get_session().get(url, timeout=45)
            r.raise_for_status()
            return r
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(1.5 * (attempt + 1))

def get_text(url, retries=4):
    return fetch(url, retries).text

def get_bytes(url, retries=4):
    return fetch(url, retries).content

def parse_record_list(text):
    return [
        line.strip().rstrip("/")
        for line in text.splitlines()
        if line.strip() and not line.strip().startswith("#")
    ]

def audit_one_segment(patient_folder, segment_name):
    if segment_name.endswith(".atr"):
        segment_name = segment_name[:-4]
    elif segment_name.endswith(".hea"):
        segment_name = segment_name[:-4]
    elif segment_name.endswith(".dat"):
        segment_name = segment_name[:-4]

    stem_url = f"{BASE_URL}/{patient_folder}/{segment_name}"
    atr_url = stem_url + ".atr"

    # WFDB reads the local annotation file; no ECG waveform is downloaded.
    local_dir = os.path.join("/content/icentia11k_annotation_cache",
                             patient_folder)
    os.makedirs(local_dir, exist_ok=True)
    local_stem = os.path.join(local_dir, segment_name)
    atr_path = local_stem + ".atr"

    if not os.path.exists(atr_path):
        with open(atr_path, "wb") as f:
            f.write(get_bytes(atr_url))

    ann = wfdb.rdann(local_stem, "atr")
    aux = [str(x).strip() for x in ann.aux_note]

    afib = sum(x == "(AFIB" for x in aux)
    nsr = sum(x == "(N" for x in aux)
    afl = sum(x == "(AFL" for x in aux)

    return {
        "patient_folder": patient_folder,
        "segment": segment_name,
        "afib_starts": afib,
        "nsr_starts": nsr,
        "afl_starts": afl,
        "annotation_count": len(ann.sample),
        "status": "ok",
        "error": ""
    }

# Load completed work from prior runs.
done_segments = set()
segment_rows, error_rows = [], []

if os.path.exists(CHECKPOINT_CSV):
    old = pd.read_csv(CHECKPOINT_CSV).fillna("")
    segment_rows = old.to_dict("records")
    done_segments = {
        (r["patient_folder"], r["segment"])
        for r in segment_rows
        if r.get("status") == "ok"
    }

if os.path.exists(ERROR_CSV):
    error_rows = pd.read_csv(ERROR_CSV).fillna("").to_dict("records")

done_patients = set()
if os.path.exists(PATIENT_CSV):
    oldp = pd.read_csv(PATIENT_CSV).fillna("")
    done_patients = set(oldp.loc[oldp["status"] == "ok",
                                 "patient_folder"].astype(str))

# Fetch authoritative patient-folder list.
top_records_url = f"{BASE_URL}/RECORDS"
patient_folders = [
    x.rstrip("/")
    for x in parse_record_list(get_text(top_records_url))
    if x.strip()
]
print("Patient folders listed:", len(patient_folders), flush=True)

# Start conservatively to avoid overloading PhysioNet.
MAX_WORKERS = 6
patients_done_this_run = 0

for patient_index, folder in enumerate(patient_folders, start=1):
    if folder in done_patients:
        continue

    try:
        patient_records_url = f"{BASE_URL}/{folder}/RECORDS"
        segment_names = [
          entry.split("/")[-1]
         for entry in parse_record_list(
            get_text(f"{BASE_URL}/{folder}/RECORDS")
          )
        ]

        if not segment_names:
            raise RuntimeError("Patient RECORDS file was empty")

        pending = []
        for name in segment_names:
            stem = name
            for suffix in (".atr", ".hea", ".dat"):
                if stem.endswith(suffix):
                    stem = stem[:-len(suffix)]
            if (folder, stem) not in done_segments:
                pending.append(stem)

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = {
                pool.submit(audit_one_segment, folder, stem): stem
                for stem in pending
            }

            for future in as_completed(futures):
                stem = futures[future]
                try:
                    row = future.result()
                    segment_rows.append(row)
                    done_segments.add((folder, stem))
                except Exception as e:
                    error_rows.append({
                        "patient_folder": folder,
                        "segment": stem,
                        "error": repr(e)
                    })

        # Persist checkpoints after every patient.
        pd.DataFrame(segment_rows).to_csv(CHECKPOINT_CSV, index=False)
        pd.DataFrame(error_rows).to_csv(ERROR_CSV, index=False)

        done_patients.add(folder)
        pd.DataFrame([
            {"patient_folder": p, "status": "ok"}
            for p in sorted(done_patients)
        ]).to_csv(PATIENT_CSV, index=False)

        patients_done_this_run += 1

        if patients_done_this_run % 10 == 0:
            afib_count = sum(
                int(r.get("afib_starts", 0) or 0) > 0
                for r in segment_rows
                if r.get("status") == "ok"
            )
            print(
                f"Patient folders processed: {len(done_patients)}/"
                f"{len(patient_folders)} | "
                f"Segments successfully audited: {len(done_segments)} | "
                f"AFib-positive segments found so far: {afib_count} | "
                f"Errors recorded: {len(error_rows)}",
                flush=True
            )

    except Exception as e:
        error_rows.append({
            "patient_folder": folder,
            "segment": "RECORDS",
            "error": repr(e)
        })
        pd.DataFrame(error_rows).to_csv(ERROR_CSV, index=False)
        print(f"ERROR reading {folder}/RECORDS: {e}", flush=True)

# Produce current AFib-positive inventory.
df = pd.DataFrame(segment_rows)
if not df.empty:
    df = df[df["status"] == "ok"].copy()
    afib_df = df[df["afib_starts"].astype(int) > 0].copy()
    afib_df.to_csv(SEGMENT_CSV, index=False)

    print("\nAUDIT STATUS")
    print("Patient folders listed:", len(patient_folders))
    print("Patient folders completed:", len(done_patients))
    print("Successfully audited segments:", len(done_segments))
    print("AFib-positive segments found:", len(afib_df))
    print("Errors logged:", len(error_rows))
    print("Checkpoint:", CHECKPOINT_CSV)
    print("AFib inventory:", SEGMENT_CSV)

Patient folders listed: 11000
Patient folders processed: 1240/11000 | Segments successfully audited: 61069 | AFib-positive segments found so far: 2807 | Errors recorded: 69
Patient folders processed: 1250/11000 | Segments successfully audited: 61560 | AFib-positive segments found so far: 2807 | Errors recorded: 69
Patient folders processed: 1260/11000 | Segments successfully audited: 62060 | AFib-positive segments found so far: 2808 | Errors recorded: 69
Patient folders processed: 1270/11000 | Segments successfully audited: 62560 | AFib-positive segments found so far: 2808 | Errors recorded: 69
Patient folders processed: 1280/11000 | Segments successfully audited: 63049 | AFib-positive segments found so far: 2858 | Errors recorded: 69
Patient folders processed: 1290/11000 | Segments successfully audited: 63546 | AFib-positive segments found so far: 2908 | Errors recorded: 72
Patient folders processed: 1300/11000 | Segments successfully audited: 64046 | AFib-positive segments found so f

Auditing Dataset to determine presence of AFib

In [ ]:
import os
import requests
import pandas as pd
import wfdb
from collections import Counter

BASE_URL = (
    "https://physionet.org/files/"
    "icentia11k-continuous-ecg/1.0"
)
LOCAL_DIR = "/content/icentia11k_audit"
os.makedirs(LOCAL_DIR, exist_ok=True)

PATIENT_IDS = range(20, 50)

def record_stem(patient_id, segment_id):
    patient = f"p{patient_id:05d}"
    group = patient[:3]
    segment = f"{patient}_s{segment_id:02d}"
    return f"{group}/{patient}/{segment}"

def download_file(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return True

    response = requests.get(url, timeout=30)
    if response.status_code != 200:
        return False

    with open(path, "wb") as f:
        f.write(response.content)
    return True

def audit_segment(patient_id, segment_id):
    stem = record_stem(patient_id, segment_id)
    local_stem = os.path.join(LOCAL_DIR, stem)
    os.makedirs(os.path.dirname(local_stem), exist_ok=True)

    # Annotation and header files are enough to audit rhythm labels.
    for ext in [".hea", ".atr"]:
        url = f"{BASE_URL}/{stem}{ext}"
        path = local_stem + ext

        if not download_file(url, path):
            return None

    ann = wfdb.rdann(local_stem, "atr")
    notes = [
        str(note).strip()
        for note in ann.aux_note
        if note is not None and str(note).strip()
    ]

    counts = Counter(notes)

    # Collect explicit rhythm-start annotations with sample positions.
    fs = wfdb.rdheader(local_stem).fs
    rhythm_events = []

    for sample, note in zip(ann.sample, ann.aux_note):
        if note is None:
            continue

        note = str(note).strip()
        if note in {"(AFIB", "(N", "(AFL"}:
            rhythm_events.append({
                "sample": int(sample),
                "time_sec": float(sample / fs),
                "rhythm": note
            })

    # Count AFIB -> explicit NSR transitions in chronological order.
    # This is a candidate count; episode boundaries still need validation.
    candidate_transitions = 0
    last_rhythm = None

    for event in rhythm_events:
        if event["rhythm"] == "(AFIB":
            last_rhythm = "(AFIB"
        elif event["rhythm"] == "(N":
            if last_rhythm == "(AFIB":
                candidate_transitions += 1
            last_rhythm = "(N"
        elif event["rhythm"] == "(AFL":
            last_rhythm = "(AFL"

    return {
        "patient_id": patient_id,
        "segment_id": segment_id,
        "n_annotations": len(ann.sample),
        "AFIB_starts": counts["(AFIB"],
        "NSR_starts_(N": counts["(N"],
        "AFL_starts": counts["(AFL"],
        "close_markers_)": counts[")"],
        "candidate_AFIB_to_NSR": candidate_transitions,
        "rhythm_annotation_counts": dict(counts),
        "events": rhythm_events
    }

# Scan sequential segment IDs for each selected patient.
# Stop after 3 consecutive missing segment headers.

rows = []
events_by_record = []
MAX_SEGMENTS = 50

for patient_id in PATIENT_IDS:
    print(f"Scanning patient {patient_id}...", flush=True)
    missing_in_a_row = 0

    for segment_id in range(MAX_SEGMENTS):
        try:
            result = audit_segment(patient_id, segment_id)
        except Exception as e:
            print(
                f"  Error at patient {patient_id}, segment {segment_id}: {e}",
                flush=True
            )
            continue

        if result is None:
            missing_in_a_row += 1
            if missing_in_a_row >= 3:
                break
            continue

        missing_in_a_row = 0
        events_by_record.append({
            "patient_id": patient_id,
            "segment_id": segment_id,
            "events": result.pop("events")
        })
        rows.append(result)

        if result["AFIB_starts"] > 0:
            print(
                f"  Found AFib: patient {patient_id}, "
                f"segment {segment_id}, "
                f"{result['AFIB_starts']} AFib starts, "
                f"{result['NSR_starts_(N']} explicit NSR starts",
                flush=True
            )

summary_df = pd.DataFrame(rows)

print("\n--- AUDIT SUMMARY ---")
print("Segments successfully audited:", len(summary_df))
print("Patients represented:", summary_df["patient_id"].nunique())

if not summary_df.empty:
    print("Total AFIB starts:", summary_df["AFIB_starts"].sum())
    print("Total explicit NSR starts:", summary_df["NSR_starts_(N"].sum())
    print(
        "Candidate AFIB-to-NSR transitions:",
        summary_df["candidate_AFIB_to_NSR"].sum()
    )

    display(
        summary_df.drop(columns=["rhythm_annotation_counts"])
        .sort_values(["patient_id", "segment_id"])
        .reset_index(drop=True)
    )


Scanning 500 annotation level data for AF presence

In [ ]:
import os
import requests
import wfdb
import pandas as pd
from io import StringIO

BASE_URL = "https://physionet.org/files/icentia11k-continuous-ecg/1.0"
LOCAL_DIR = "/content/icentia11k_audit"
os.makedirs(LOCAL_DIR, exist_ok=True)

def get_patient_segments(patient_id):
    patient = f"p{patient_id:05d}"
    folder = f"{patient[:3]}/{patient}"
    url = f"{BASE_URL}/{folder}/RECORDS"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    # Each entry should identify a segment stem, e.g. p00060_s00
    stems = [
        line.strip()
        for line in response.text.splitlines()
        if line.strip() and not line.strip().startswith("#")
    ]

    # Normalize entries to filenames/stems if the list contains paths
    stems = [s.split("/")[-1] for s in stems]
    stems = [s[:-4] if s.endswith(".atr") else s for s in stems]

    return folder, stems

def audit_patient(patient_id):
    folder, stems = get_patient_segments(patient_id)
    results = []
    errors = []

    for i, stem_name in enumerate(stems, start=1):
        stem = f"{folder}/{stem_name}"
        local_stem = os.path.join(LOCAL_DIR, stem)
        os.makedirs(os.path.dirname(local_stem), exist_ok=True)

        atr_url = f"{BASE_URL}/{stem}.atr"
        atr_path = local_stem + ".atr"

        try:
            if not os.path.exists(atr_path):
                r = requests.get(atr_url, timeout=30)
                r.raise_for_status()
                with open(atr_path, "wb") as f:
                    f.write(r.content)

            ann = wfdb.rdann(local_stem, "atr")

            aux = [str(x).strip() for x in ann.aux_note]
            afib_count = sum(x == "(AFIB" for x in aux)
            nsr_count = sum(x == "(N" for x in aux)
            afl_count = sum(x == "(AFL" for x in aux)

            if afib_count:
                results.append({
                    "patient_id": patient_id,
                    "segment": stem_name,
                    "afib_starts": afib_count,
                    "nsr_starts": nsr_count,
                    "afl_starts": afl_count,
                    "annotation_count": len(ann.sample),
                })

        except Exception as e:
            errors.append({
                "patient_id": patient_id,
                "segment": stem_name,
                "error": str(e),
            })

        if i % 10 == 0 or i == len(stems):
            print(
                f"Patient {patient_id}: {i}/{len(stems)} segments checked",
                flush=True
            )

    return results, errors, len(stems)

# Test the complete workflow on patient 60
results, errors, n_segments = audit_patient(60)

print("\nSegments listed:", n_segments)
print("AFib-positive segments:", len(results))
print("Errors:", len(errors))

display(pd.DataFrame(results))
if errors:
    display(pd.DataFrame(errors))

Patient 60: 10/50 segments checked
Patient 60: 20/50 segments checked
Patient 60: 30/50 segments checked
Patient 60: 40/50 segments checked
Patient 60: 50/50 segments checked

Segments listed: 50
AFib-positive segments: 50
Errors: 0


,patient_id,segment,afib_starts,nsr_starts,afl_starts,annotation_count
0,60,p00060_s00,59,0,0,6090
1,60,p00060_s01,60,0,0,6473
2,60,p00060_s02,67,0,0,6659
3,60,p00060_s03,22,0,0,5869
4,60,p00060_s04,70,0,0,6511
5,60,p00060_s05,70,0,0,6744
6,60,p00060_s06,70,0,0,6276
7,60,p00060_s07,74,0,0,6588
8,60,p00060_s08,76,0,0,6113
9,60,p00060_s09,35,0,0,5938


In [ ]:

import requests

url = "https://physionet.org/files/icentia11k-continuous-ecg/1.0/RECORDS"
r = requests.get(url, timeout=30)

print("HTTP status:", r.status_code)
print("Bytes:", len(r.content))
print("First 20 lines:")
print("\n".join(r.text.splitlines()[:20]))


HTTP status: 200
Bytes: 132000
First 20 lines:
p00/p00000/
p00/p00001/
p00/p00002/
p00/p00003/
p00/p00004/
p00/p00005/
p00/p00006/
p00/p00007/
p00/p00008/
p00/p00009/
p00/p00010/
p00/p00011/
p00/p00012/
p00/p00013/
p00/p00014/
p00/p00015/
p00/p00016/
p00/p00017/
p00/p00018/
p00/p00019/
